<a href="https://colab.research.google.com/github/XTMay/LLM_AI_Agent/blob/main/Notebook/LLM_RMSNORM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM - Encoder_Decoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy
import numpy as np
from typing import Optional, Tuple

In [ ]:
class TransformerConfig:
    """Transformer配置"""
    def __init__(
        self,
        src_vocab_size: int = 10000,
        tgt_vocab_size: int = 10000,
        d_model: int = 512,          # 模型维度
        n_head: int = 8,             # 多头注意力头数
        n_layers: int = 6,           # 编码器和解码器层数
        d_ff: int = 2048,            # 前馈网络维度
        max_seq_length: int = 512,   # 最大序列长度
        dropout: float = 0.1,        # Dropout率
        label_smoothing: float = 0.1, # 标签平滑
        pad_idx: int = 0,            # Padding token索引
    ):
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.d_model = d_model
        self.n_head = n_head
        self.n_layers = n_layers
        self.d_ff = d_ff
        self.max_seq_length = max_seq_length
        self.dropout = dropout
        self.label_smoothing = label_smoothing
        self.pad_idx = pad_idx

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    多头注意力机制 - 论文核心创新
    """
    def __init__(self, d_model: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_head == 0

        self.d_model = d_model
        self.n_head = n_head
        self.d_k = d_model // n_head

        # 线性变换层 - 论文中的W^Q, W^K, W^V
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)  # 输出投影 W^O

        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, Q: torch.Tensor, K: torch.Tensor,
                                   V: torch.Tensor, mask: Optional[torch.Tensor] = None):
        """
        缩放点积注意力 - 论文公式
        Attention(Q,K,V) = softmax(QK^T / sqrt(d_k))V
        """
        # 计算注意力分数
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 应用掩码（如果有）
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # Softmax归一化
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        # 应用注意力权重
        context = torch.matmul(attention_weights, V)

        return context, attention_weights

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
                mask: Optional[torch.Tensor] = None):
        batch_size = query.size(0)
        seq_len = query.size(1)
        key_len = key.size(1)

        # 1. 线性变换得到Q, K, V
        Q = self.w_q(query)  # [batch_size, seq_len, d_model]
        K = self.w_k(key)    # [batch_size, key_len, d_model]
        V = self.w_v(value)  # [batch_size, key_len, d_model]

        # 2. 重塑为多头格式
        Q = Q.view(batch_size, seq_len, self.n_head, self.d_k).transpose(1, 2)
        K = K.view(batch_size, key_len, self.n_head, self.d_k).transpose(1, 2)
        V = V.view(batch_size, key_len, self.n_head, self.d_k).transpose(1, 2)

        # 3. 处理掩码
        if mask is not None:
            # 确保mask是4维的
            while mask.dim() < 4:
                mask = mask.unsqueeze(0)

            # 如果第二个维度是1，扩展到多头
            if mask.size(1) == 1:
                mask = mask.expand(-1, self.n_head, -1, -1)

        # 4. 计算缩放点积注意力
        context, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)

        # 5. 重塑回原始维度
        context = context.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_model
        )

        # 6. 输出投影
        output = self.w_o(context)

        return output, attention_weights

In [ ]:
class PositionalEncoding(nn.Module):
    """
    位置编码 - 论文中的正弦余弦位置编码
    """
    def __init__(self, d_model: int, max_seq_length: int = 5000):
        super().__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                           (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1)]

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """逐位置前馈网络"""
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

In [ ]:
class EncoderLayer(nn.Module):
    """编码器层"""
    def __init__(self, d_model: int, n_head: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_head, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor] = None):
        attn_output, _ = self.self_attention(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

In [ ]:
class EncoderLayer(nn.Module):
    """编码器层"""
    def __init__(self, d_model: int, n_head: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_head, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor] = None):
        attn_output, _ = self.self_attention(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

In [ ]:
class DecoderLayer(nn.Module):
    """解码器层"""
    def __init__(self, d_model: int, n_head: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_head, dropout)
        self.cross_attention = MultiHeadAttention(d_model, n_head, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None):
        # 掩码自注意力
        self_attn_output, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_output))

        # 编码器-解码器注意力
        cross_attn_output, _ = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))

        # 前馈网络
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x

In [ ]:
class Encoder(nn.Module):
    """编码器"""
    def __init__(self, layer: EncoderLayer, n_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(layer.self_attention.d_model)

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor] = None):
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)

In [ ]:
class Decoder(nn.Module):
    """解码器"""
    def __init__(self, layer: DecoderLayer, n_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(layer.self_attention.d_model)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

In [ ]:
class Embeddings(nn.Module):
    """嵌入层 + 位置编码"""
    def __init__(self, vocab_size: int, d_model: int, max_seq_length: int = 512):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.positional_encoding(self.embedding(x) * math.sqrt(self.d_model))

In [ ]:
class Generator(nn.Module):
    """输出生成器"""
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.projection = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.log_softmax(self.projection(x), dim=-1)

In [ ]:

class Transformer(nn.Module):
    """完整的Transformer模型"""
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.config = config

        encoder_layer = EncoderLayer(config.d_model, config.n_head, config.d_ff, config.dropout)
        decoder_layer = DecoderLayer(config.d_model, config.n_head, config.d_ff, config.dropout)

        self.src_embedding = Embeddings(config.src_vocab_size, config.d_model, config.max_seq_length)
        self.tgt_embedding = Embeddings(config.tgt_vocab_size, config.d_model, config.max_seq_length)

        self.encoder = Encoder(encoder_layer, config.n_layers)
        self.decoder = Decoder(decoder_layer, config.n_layers)

        self.generator = Generator(config.d_model, config.tgt_vocab_size)

        self.init_parameters()

    def init_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None):
        src_emb = self.src_embedding(src)
        return self.encoder(src_emb, src_mask)

    def decode(self, tgt: torch.Tensor, encoder_output: torch.Tensor,
               src_mask: Optional[torch.Tensor] = None,
               tgt_mask: Optional[torch.Tensor] = None):
        tgt_emb = self.tgt_embedding(tgt)
        return self.decoder(tgt_emb, encoder_output, src_mask, tgt_mask)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None):
        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)
        return self.generator(decoder_output)

In [ ]:
def create_padding_mask(seq: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """创建padding掩码"""
    return (seq != pad_idx).unsqueeze(1).unsqueeze(1)

In [ ]:
def create_look_ahead_mask(size: int, device: torch.device) -> torch.Tensor:
    """创建look-ahead掩码"""
    mask = torch.tril(torch.ones(size, size, device=device))
    return mask.unsqueeze(0).unsqueeze(0)

In [ ]:
def create_target_mask(tgt: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """创建目标序列掩码"""
    batch_size, tgt_len = tgt.shape
    device = tgt.device

    # Padding掩码
    tgt_padding_mask = create_padding_mask(tgt, pad_idx)

    # Look-ahead掩码
    look_ahead_mask = create_look_ahead_mask(tgt_len, device)

    # 组合掩码 - 使用乘法而不是位运算
    tgt_mask = tgt_padding_mask.float() * look_ahead_mask.float()

    return tgt_mask

In [ ]:
def test_transformer_model():
    """测试Transformer模型"""
    print("=== 经典Transformer模型测试 ===")

    config = TransformerConfig(
        src_vocab_size=1000,
        tgt_vocab_size=1000,
        d_model=512,
        n_head=8,
        n_layers=6,
        d_ff=2048,
        max_seq_length=100,
        dropout=0.1
    )

    model = Transformer(config)

    batch_size = 2
    src_seq_len = 20
    tgt_seq_len = 15

    src = torch.randint(1, config.src_vocab_size, (batch_size, src_seq_len))
    tgt = torch.randint(1, config.tgt_vocab_size, (batch_size, tgt_seq_len))

    # 创建掩码
    src_mask = create_padding_mask(src, config.pad_idx)
    tgt_input = tgt[:, :-1]
    tgt_mask = create_target_mask(tgt_input, config.pad_idx)

    print(f"源序列形状: {src.shape}")
    print(f"目标序列形状: {tgt.shape}")
    print(f"解码器输入形状: {tgt_input.shape}")
    print(f"源掩码形状: {src_mask.shape}")
    print(f"目标掩码形状: {tgt_mask.shape}")
    print(f"模型参数数量: {sum(p.numel() for p in model.parameters()):,}")

    # 前向传播
    output = model(src, tgt_input, src_mask, tgt_mask)

    print(f"输出形状: {output.shape}")

    # 检查log概率
    prob_sums = torch.exp(output).sum(dim=-1)
    print(f"概率和的范围: [{prob_sums.min():.6f}, {prob_sums.max():.6f}]")
    print(f"概率和接近1: {torch.allclose(prob_sums, torch.ones_like(prob_sums), atol=1e-5)}")

    return model

In [ ]:
def test_attention_mechanism():
    """测试注意力机制"""
    print("\n=== 多头注意力机制测试 ===")

    d_model = 512
    n_head = 8
    seq_len = 10
    batch_size = 2

    attention = MultiHeadAttention(d_model, n_head)
    x = torch.randn(batch_size, seq_len, d_model)

    output, weights = attention(x, x, x)

    print(f"输入形状: {x.shape}")
    print(f"输出形状: {output.shape}")
    print(f"注意力权重形状: {weights.shape}")

    # 验证注意力权重归一化
    print(f"注意力权重和为1: {torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)))}")

    return attention, weights

In [ ]:
def test_positional_encoding():
    """测试位置编码"""
    print("\n=== 位置编码测试 ===")

    d_model = 512
    max_seq_length = 100

    pe = PositionalEncoding(d_model, max_seq_length)
    x = torch.randn(2, 20, d_model)
    x_with_pe = pe(x)

    print(f"原始输入形状: {x.shape}")
    print(f"位置编码后形状: {x_with_pe.shape}")

    # 可视化位置编码
    pe_matrix = pe.pe[0, :20, :10]
    print(f"位置编码矩阵形状: {pe_matrix.shape}")
    print("位置编码前几个位置的值:")
    print(pe_matrix[:5, :5])

    return pe

In [ ]:
if __name__ == "__main__":
    print("🚀 经典Transformer实现 - 'Attention Is All You Need'")
    print("=" * 60)

    torch.manual_seed(42)

    # 1. 测试完整模型
    transformer_model = test_transformer_model()

    # 2. 测试注意力机制
    attention_module, attention_weights = test_attention_mechanism()

    # 3. 测试位置编码
    pos_encoding = test_positional_encoding()

    print("\n" + "=" * 60)
    print("🎯 经典Transformer实现总结")
    print("=" * 60)
    print("✅ 完整的Encoder-Decoder架构")
    print("✅ 多头注意力机制 (核心创新)")
    print("✅ 正弦余弦位置编码")
    print("✅ 残差连接 + LayerNorm")
    print("✅ 位置相关前馈网络")
    print("✅ 掩码机制（修复版本）")

    param_count = sum(p.numel() for p in transformer_model.parameters())
    print(f"\n📊 模型统计:")
    print(f"- 参数数量: {param_count:,}")
    print(f"- 论文原始模型: ~65M 参数")
    print(f"- 核心创新: 完全基于注意力机制，摒弃递归和卷积")

    print(f"\n🔬 关键技术贡献:")
    print(f"- 提出了Scaled Dot-Product Attention")
    print(f"- 证明了注意力机制的并行化优势")
    print(f"- 建立了现代Transformer架构的基础")
    print(f"- 为BERT、GPT等模型奠定了理论基础")

🚀 经典Transformer实现 - 'Attention Is All You Need'
=== 经典Transformer模型测试 ===
源序列形状: torch.Size([2, 20])
目标序列形状: torch.Size([2, 15])
解码器输入形状: torch.Size([2, 14])
源掩码形状: torch.Size([2, 1, 1, 20])
目标掩码形状: torch.Size([2, 1, 14, 14])
模型参数数量: 45,677,544
输出形状: torch.Size([2, 14, 1000])
概率和的范围: [1.000000, 1.000000]
概率和接近1: True

=== 多头注意力机制测试 ===
输入形状: torch.Size([2, 10, 512])
输出形状: torch.Size([2, 10, 512])
注意力权重形状: torch.Size([2, 8, 10, 10])
注意力权重和为1: False

=== 位置编码测试 ===
原始输入形状: torch.Size([2, 20, 512])
位置编码后形状: torch.Size([2, 20, 512])
位置编码矩阵形状: torch.Size([20, 10])
位置编码前几个位置的值:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000],
        [ 0.8415,  0.5403,  0.8219,  0.5697,  0.8020],
        [ 0.9093, -0.4161,  0.9364, -0.3509,  0.9581],
        [ 0.1411, -0.9900,  0.2451, -0.9695,  0.3428],
        [-0.7568, -0.6536, -0.6572, -0.7537, -0.5486]])

🎯 经典Transformer实现总结
✅ 完整的Encoder-Decoder架构
✅ 多头注意力机制 (核心创新)
✅ 正弦余弦位置编码
✅ 残差连接 + LayerNorm
✅ 位置相关前馈网络
✅ 掩码机制（修复版本）

📊 模型统计:
- 参数数量: 45,677,544
- 

# RMS-NORM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy
import numpy as np
from typing import Optional, Tuple

In [ ]:
class TransformerConfig:
    """Transformer配置"""
    def __init__(
        self,
        src_vocab_size: int = 10000,
        tgt_vocab_size: int = 10000,
        d_model: int = 512,          # 模型维度
        n_head: int = 8,             # 多头注意力头数
        n_layers: int = 6,           # 编码器和解码器层数
        d_ff: int = 2048,            # 前馈网络维度
        max_seq_length: int = 512,   # 最大序列长度
        dropout: float = 0.1,        # Dropout率
        label_smoothing: float = 0.1, # 标签平滑
        pad_idx: int = 0,            # Padding token索引
    ):
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.d_model = d_model
        self.n_head = n_head
        self.n_layers = n_layers
        self.d_ff = d_ff
        self.max_seq_length = max_seq_length
        self.dropout = dropout
        self.label_smoothing = label_smoothing
        self.pad_idx = pad_idx

In [ ]:
class RMSNorm(nn.Module):
    """
    RMS Normalization - 现代LLM中常用的归一化方法
    相比LayerNorm，RMSNorm去掉了减均值的操作，只保留缩放
    公式: RMSNorm(x) = x / RMS(x) * g
    其中 RMS(x) = sqrt(mean(x^2) + eps)

    优势:
    1. 计算更高效 - 不需要计算均值和方差
    2. 数值更稳定 - 避免了减均值操作
    3. 现代模型采用 - LLaMA、PaLM等都使用RMSNorm
    """
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 计算RMS值
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
        # 归一化并缩放
        return x / rms * self.weight

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    多头注意力机制 - 论文核心创新
    """
    def __init__(self, d_model: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_head == 0

        self.d_model = d_model
        self.n_head = n_head
        self.d_k = d_model // n_head

        # 线性变换层 - 论文中的W^Q, W^K, W^V
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)  # 输出投影 W^O

        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, Q: torch.Tensor, K: torch.Tensor,
                                   V: torch.Tensor, mask: Optional[torch.Tensor] = None):
        """
        缩放点积注意力 - 论文公式
        Attention(Q,K,V) = softmax(QK^T / sqrt(d_k))V
        """
        # 计算注意力分数
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 应用掩码（如果有）
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # Softmax归一化
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        # 应用注意力权重
        context = torch.matmul(attention_weights, V)

        return context, attention_weights

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
                mask: Optional[torch.Tensor] = None):
        batch_size = query.size(0)
        seq_len = query.size(1)
        key_len = key.size(1)

        # 1. 线性变换得到Q, K, V
        Q = self.w_q(query)  # [batch_size, seq_len, d_model]
        K = self.w_k(key)    # [batch_size, key_len, d_model]
        V = self.w_v(value)  # [batch_size, key_len, d_model]

        # 2. 重塑为多头格式
        Q = Q.view(batch_size, seq_len, self.n_head, self.d_k).transpose(1, 2)
        K = K.view(batch_size, key_len, self.n_head, self.d_k).transpose(1, 2)
        V = V.view(batch_size, key_len, self.n_head, self.d_k).transpose(1, 2)

        # 3. 处理掩码
        if mask is not None:
            # 确保mask是4维的
            while mask.dim() < 4:
                mask = mask.unsqueeze(0)

            # 如果第二个维度是1，扩展到多头
            if mask.size(1) == 1:
                mask = mask.expand(-1, self.n_head, -1, -1)

        # 4. 计算缩放点积注意力
        context, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)

        # 5. 重塑回原始维度
        context = context.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_model
        )

        # 6. 输出投影
        output = self.w_o(context)

        return output, attention_weights

In [ ]:
class PositionalEncoding(nn.Module):
    """
    位置编码 - 论文中的正弦余弦位置编码
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model: int, max_seq_length: int = 5000):
        super().__init__()

        # 创建位置编码矩阵
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)

        # 计算除数项
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                           (-math.log(10000.0) / d_model))

        # 应用正弦和余弦
        pe[:, 0::2] = torch.sin(position * div_term)  # 偶数位置
        pe[:, 1::2] = torch.cos(position * div_term)  # 奇数位置

        # 注册为buffer，不参与梯度更新
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """添加位置编码到输入嵌入"""
        return x + self.pe[:, :x.size(1)]

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """
    逐位置前馈网络 - 论文中的FFN
    FFN(x) = max(0, xW1 + b1)W2 + b2
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """前馈网络前向传播"""
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

In [ ]:
class EncoderLayer(nn.Module):
    """
    编码器层 - 论文图1左侧
    包含多头自注意力和前馈网络，每个子层都有残差连接和RMSNorm
    """
    def __init__(self, d_model: int, n_head: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_head, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = RMSNorm(d_model)  # 使用RMSNorm替代LayerNorm
        self.norm2 = RMSNorm(d_model)  # 使用RMSNorm替代LayerNorm
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor] = None):
        """编码器层前向传播"""
        # 1. 多头自注意力 + 残差连接 + RMSNorm
        attn_output, _ = self.self_attention(x, x, x, src_mask)
        x = self.norm1(x + self.dropout(attn_output))

        # 2. 前馈网络 + 残差连接 + RMSNorm
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))

        return x

In [ ]:
class DecoderLayer(nn.Module):
    """
    解码器层 - 论文图1右侧
    包含掩码多头自注意力、编码器-解码器注意力和前馈网络
    """
    def __init__(self, d_model: int, n_head: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, n_head, dropout)
        self.cross_attention = MultiHeadAttention(d_model, n_head, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = RMSNorm(d_model)  # 使用RMSNorm替代LayerNorm
        self.norm2 = RMSNorm(d_model)  # 使用RMSNorm替代LayerNorm
        self.norm3 = RMSNorm(d_model)  # 使用RMSNorm替代LayerNorm
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None):
        """解码器层前向传播"""
        # 1. 掩码多头自注意力 + 残差连接 + RMSNorm
        self_attn_output, _ = self.self_attention(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_output))

        # 2. 编码器-解码器注意力 + 残差连接 + RMSNorm
        cross_attn_output, _ = self.cross_attention(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))

        # 3. 前馈网络 + 残差连接 + RMSNorm
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))

        return x

In [ ]:
class Encoder(nn.Module):
    """
    编码器 - N个相同编码器层的堆叠
    """
    def __init__(self, layer: EncoderLayer, n_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(n_layers)])
        self.norm = RMSNorm(layer.self_attention.d_model)  # 使用RMSNorm替代LayerNorm

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor] = None):
        """编码器前向传播"""
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)

In [ ]:
class Decoder(nn.Module):
    """
    解码器 - N个相同解码器层的堆叠
    """
    def __init__(self, layer: DecoderLayer, n_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(n_layers)])
        self.norm = RMSNorm(layer.self_attention.d_model)  # 使用RMSNorm替代LayerNorm

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None):
        """解码器前向传播"""
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

In [ ]:
class Embeddings(nn.Module):
    """
    嵌入层 + 位置编码
    论文中提到的嵌入需要乘以sqrt(d_model)
    """
    def __init__(self, vocab_size: int, d_model: int, max_seq_length: int = 512):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """嵌入 + 位置编码"""
        return self.positional_encoding(self.embedding(x) * math.sqrt(self.d_model))

In [ ]:
class Generator(nn.Module):
    """
    输出生成器 - 将解码器输出转换为词汇表概率
    """
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.projection = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """生成词汇表概率分布"""
        return F.log_softmax(self.projection(x), dim=-1)

In [ ]:
class Transformer(nn.Module):
    """
    完整的Transformer模型 - "Attention Is All You Need"
    """
    def __init__(self, config: TransformerConfig):
        super().__init__()
        self.config = config

        # 创建编码器和解码器层
        encoder_layer = EncoderLayer(config.d_model, config.n_head, config.d_ff, config.dropout)
        decoder_layer = DecoderLayer(config.d_model, config.n_head, config.d_ff, config.dropout)

        # 嵌入层
        self.src_embedding = Embeddings(config.src_vocab_size, config.d_model, config.max_seq_length)
        self.tgt_embedding = Embeddings(config.tgt_vocab_size, config.d_model, config.max_seq_length)

        # 编码器和解码器
        self.encoder = Encoder(encoder_layer, config.n_layers)
        self.decoder = Decoder(decoder_layer, config.n_layers)

        # 输出生成器
        self.generator = Generator(config.d_model, config.tgt_vocab_size)

        # 初始化参数
        self.init_parameters()

    def init_parameters(self):
        """初始化模型参数 - 论文中提到的Xavier初始化"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src: torch.Tensor, src_mask: Optional[torch.Tensor] = None):
        """编码源序列"""
        src_emb = self.src_embedding(src)
        return self.encoder(src_emb, src_mask)

    def decode(self, tgt: torch.Tensor, encoder_output: torch.Tensor,
               src_mask: Optional[torch.Tensor] = None,
               tgt_mask: Optional[torch.Tensor] = None):
        """解码目标序列"""
        tgt_emb = self.tgt_embedding(tgt)
        return self.decoder(tgt_emb, encoder_output, src_mask, tgt_mask)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor,
                src_mask: Optional[torch.Tensor] = None,
                tgt_mask: Optional[torch.Tensor] = None):
        """完整的前向传播"""
        # 编码
        encoder_output = self.encode(src, src_mask)

        # 解码
        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)

        # 生成输出概率
        return self.generator(decoder_output)

In [ ]:
class MaskGenerator:
    """
    掩码生成器 - 用于创建各种类型的注意力掩码
    """
    @staticmethod
    def create_padding_mask(seq: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
        """创建padding掩码 - 返回 [batch_size, 1, 1, seq_len] 格式"""
        # [batch_size, seq_len] -> [batch_size, 1, 1, seq_len]
        return (seq != pad_idx).unsqueeze(1).unsqueeze(2)

    @staticmethod
    def create_look_ahead_mask(size: int, device: torch.device) -> torch.Tensor:
        """创建look-ahead掩码（下三角矩阵）- 返回 [1, 1, size, size] 格式"""
        mask = torch.tril(torch.ones(size, size, device=device))
        return mask.unsqueeze(0).unsqueeze(0)  # [1, 1, size, size]

    @staticmethod
    def create_target_mask(tgt: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
        """创建目标序列掩码（padding + look-ahead）"""
        batch_size, tgt_len = tgt.shape
        device = tgt.device

        # Padding掩码 [batch_size, 1, 1, tgt_len]
        tgt_padding_mask = MaskGenerator.create_padding_mask(tgt, pad_idx)

        # Look-ahead掩码 [1, 1, tgt_len, tgt_len]
        look_ahead_mask = MaskGenerator.create_look_ahead_mask(tgt_len, device)

        # 扩展padding掩码的维度以便广播
        # [batch_size, 1, 1, tgt_len] -> [batch_size, 1, tgt_len, tgt_len]
        tgt_padding_mask = tgt_padding_mask.expand(-1, -1, tgt_len, -1)

        # 组合掩码：两个掩码都必须为True（使用乘法避免位运算错误）
        tgt_mask = tgt_padding_mask.float() * look_ahead_mask.float()

        return tgt_mask

In [ ]:
class LabelSmoothingLoss(nn.Module):
    """
    标签平滑损失 - 论文中提到的正则化技术
    """
    def __init__(self, size: int, padding_idx: int, smoothing: float = 0.0):
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction='sum')
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size

    def forward(self, x: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """计算标签平滑损失"""
        assert x.size(1) == self.size
        true_dist = x.data.clone()
        true_dist.fill_(self.smoothing / (self.size - 2))
        true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        mask = torch.nonzero(target.data == self.padding_idx, as_tuple=False)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        return self.criterion(x, true_dist)

In [ ]:
class TransformerTrainer:
    """
    Transformer训练器
    """
    def __init__(self, model: Transformer, config: TransformerConfig):
        self.model = model
        self.config = config
        self.criterion = LabelSmoothingLoss(
            config.tgt_vocab_size, config.pad_idx, config.label_smoothing
        )

    def create_optimizer(self, lr: float = 1.0, betas: Tuple[float, float] = (0.9, 0.98), eps: float = 1e-9):
        """创建优化器 - 论文中使用的Adam优化器"""
        return torch.optim.Adam(self.model.parameters(), lr=lr, betas=betas, eps=eps)

    def lr_scheduler(self, optimizer, step_num: int, warmup_steps: int = 4000):
        """学习率调度器 - 论文中的预热学习率调度"""
        d_model = self.config.d_model
        step_num = max(step_num, 1)  # 避免除零
        lr = (d_model ** -0.5) * min(step_num ** -0.5, step_num * (warmup_steps ** -1.5))

        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        return lr

    def train_step(self, src: torch.Tensor, tgt: torch.Tensor, optimizer, step_num: int):
        """单步训练"""
        self.model.train()

        # 创建掩码
        src_mask = MaskGenerator.create_padding_mask(src, self.config.pad_idx)
        tgt_mask = MaskGenerator.create_target_mask(tgt[:, :-1], self.config.pad_idx)

        # 前向传播
        output = self.model(src, tgt[:, :-1], src_mask, tgt_mask)

        # 计算损失
        loss = self.criterion(output.contiguous().view(-1, output.size(-1)),
                            tgt[:, 1:].contiguous().view(-1))

        # 反向传播
        optimizer.zero_grad()
        loss.backward()

        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

        # 更新学习率
        self.lr_scheduler(optimizer, step_num)

        # 参数更新
        optimizer.step()

        return loss.item()

In [ ]:
class TransformerTranslator:
    """
    Transformer翻译器 - 用于推理时的文本生成
    """
    def __init__(self, model: Transformer, config: TransformerConfig):
        self.model = model
        self.config = config

    def greedy_decode(self, src: torch.Tensor, src_mask: torch.Tensor,
                     max_len: int, start_symbol: int, end_symbol: int):
        """贪心解码"""
        self.model.eval()

        # 编码源序列
        encoder_output = self.model.encode(src, src_mask)

        # 初始化解码器输入
        batch_size = src.size(0)
        device = src.device
        ys = torch.ones(batch_size, 1, device=device, dtype=torch.long).fill_(start_symbol)

        with torch.no_grad():
            for i in range(max_len - 1):
                # 创建目标掩码
                tgt_mask = MaskGenerator.create_target_mask(ys, self.config.pad_idx)

                # 解码
                out = self.model.decode(ys, encoder_output, src_mask, tgt_mask)

                # 生成概率
                prob = self.model.generator(out[:, -1])

                # 选择最大概率的词
                _, next_word = torch.max(prob, dim=1)
                next_word = next_word.unsqueeze(1)

                # 添加到序列
                ys = torch.cat([ys, next_word], dim=1)

                # 检查是否所有序列都结束
                if (next_word == end_symbol).all():
                    break

        return ys

In [ ]:
def test_transformer_model():
    """测试Transformer模型"""
    print("=== 现代化Transformer模型测试（使用RMSNorm）===")

    # 创建配置
    config = TransformerConfig(
        src_vocab_size=1000,
        tgt_vocab_size=1000,
        d_model=512,
        n_head=8,
        n_layers=6,
        d_ff=2048,
        max_seq_length=100,
        dropout=0.1
    )

    # 创建模型
    model = Transformer(config)

    # 测试数据
    batch_size = 2
    src_seq_len = 20
    tgt_seq_len = 15

    src = torch.randint(1, config.src_vocab_size, (batch_size, src_seq_len))
    tgt = torch.randint(1, config.tgt_vocab_size, (batch_size, tgt_seq_len))

    # 创建掩码
    src_mask = MaskGenerator.create_padding_mask(src, config.pad_idx)
    tgt_input = tgt[:, :-1]  # 解码器输入（去掉最后一个token）
    tgt_mask = MaskGenerator.create_target_mask(tgt_input, config.pad_idx)

    print(f"源序列形状: {src.shape}")
    print(f"目标序列形状: {tgt.shape}")
    print(f"解码器输入形状: {tgt_input.shape}")
    print(f"源掩码形状: {src_mask.shape}")
    print(f"目标掩码形状: {tgt_mask.shape}")
    print(f"模型参数数量: {sum(p.numel() for p in model.parameters()):,}")

    # 前向传播
    output = model(src, tgt_input, src_mask, tgt_mask)

    print(f"输出形状: {output.shape}")
    print(f"输出是log概率: 检查概率和是否为1...")

    # 检查log概率的指数和是否接近1
    prob_sums = torch.exp(output).sum(dim=-1)
    print(f"概率和的范围: [{prob_sums.min():.6f}, {prob_sums.max():.6f}]")
    print(f"概率和接近1: {torch.allclose(prob_sums, torch.ones_like(prob_sums), atol=1e-5)}")

    return model

In [ ]:
def test_attention_mechanism():
    """测试注意力机制"""
    print("\n=== 多头注意力机制测试 ===")

    d_model = 512
    n_head = 8
    seq_len = 10
    batch_size = 2

    # 创建注意力模块
    attention = MultiHeadAttention(d_model, n_head)

    # 测试输入
    x = torch.randn(batch_size, seq_len, d_model)

    # 自注意力
    output, weights = attention(x, x, x)

    print(f"输入形状: {x.shape}")
    print(f"输出形状: {output.shape}")
    print(f"注意力权重形状: {weights.shape}")

    # 验证注意力权重归一化
    print(f"注意力权重和为1: {torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)))}")

    return attention, weights

In [ ]:
def test_positional_encoding():
    """测试位置编码"""
    print("\n=== 位置编码测试 ===")

    d_model = 512
    max_seq_length = 100

    # 创建位置编码
    pe = PositionalEncoding(d_model, max_seq_length)

    # 测试输入
    x = torch.randn(2, 20, d_model)
    x_with_pe = pe(x)

    print(f"原始输入形状: {x.shape}")
    print(f"位置编码后形状: {x_with_pe.shape}")

    # 可视化位置编码模式
    pe_matrix = pe.pe[0, :20, :10]  # 取前20个位置，前10个维度
    print(f"位置编码矩阵形状: {pe_matrix.shape}")
    print("位置编码前几个位置的值:")
    print(pe_matrix[:5, :5])

    return pe

In [ ]:
def test_rmsnorm_vs_layernorm():
    """测试RMSNorm与LayerNorm的对比"""
    print("\n=== RMSNorm vs LayerNorm 对比测试 ===")

    d_model = 512
    batch_size = 2
    seq_len = 10

    # 创建测试数据
    x = torch.randn(batch_size, seq_len, d_model)

    # RMSNorm
    rms_norm = RMSNorm(d_model)

    # LayerNorm（用于对比）
    layer_norm = nn.LayerNorm(d_model)

    # 前向传播
    rms_output = rms_norm(x)
    layer_output = layer_norm(x)

    print(f"输入形状: {x.shape}")
    print(f"RMSNorm输出形状: {rms_output.shape}")
    print(f"LayerNorm输出形状: {layer_output.shape}")

    # 计算统计信息
    print(f"输入均值: {x.mean():.6f}, 方差: {x.var():.6f}")
    print(f"RMSNorm输出均值: {rms_output.mean():.6f}, 方差: {rms_output.var():.6f}")
    print(f"LayerNorm输出均值: {layer_output.mean():.6f}, 方差: {layer_output.var():.6f}")

    # 参数数量对比
    rms_params = sum(p.numel() for p in rms_norm.parameters())
    layer_params = sum(p.numel() for p in layer_norm.parameters())

    print(f"RMSNorm参数数量: {rms_params}")
    print(f"LayerNorm参数数量: {layer_params}")
    print(f"参数减少比例: {(layer_params - rms_params) / layer_params * 100:.1f}%")

    return rms_norm, layer_norm

In [ ]:
def test_training_process():
    """测试训练过程"""
    print("\n=== 训练过程测试 ===")

    # 创建小型模型用于快速测试
    config = TransformerConfig(
        src_vocab_size=100,
        tgt_vocab_size=100,
        d_model=128,
        n_head=4,
        n_layers=2,
        d_ff=256,
        max_seq_length=50,
        dropout=0.1,
        label_smoothing=0.1
    )

    model = Transformer(config)
    trainer = TransformerTrainer(model, config)
    optimizer = trainer.create_optimizer(lr=1.0)

    # 模拟训练数据
    batch_size = 4
    src_len = 10
    tgt_len = 8

    src = torch.randint(1, config.src_vocab_size, (batch_size, src_len))
    tgt = torch.randint(1, config.tgt_vocab_size, (batch_size, tgt_len))

    print(f"训练数据 - 源序列: {src.shape}, 目标序列: {tgt.shape}")

    # 训练几步
    losses = []
    for step in range(1, 6):
        loss = trainer.train_step(src, tgt, optimizer, step)
        losses.append(loss)
        print(f"步骤 {step}: 损失 = {loss:.4f}")

    print(f"损失变化: {losses}")

    return trainer, losses

In [ ]:
def test_translation_inference():
    """测试翻译推理"""
    print("\n=== 翻译推理测试 ===")

    config = TransformerConfig(
        src_vocab_size=100,
        tgt_vocab_size=100,
        d_model=128,
        n_head=4,
        n_layers=2,
        d_ff=256,
        max_seq_length=20,
    )

    model = Transformer(config)
    translator = TransformerTranslator(model, config)

    # 测试序列
    batch_size = 2
    src_len = 8
    src = torch.randint(1, config.src_vocab_size, (batch_size, src_len))
    src_mask = MaskGenerator.create_padding_mask(src, config.pad_idx)

    print(f"源序列: {src}")
    print(f"源掩码形状: {src_mask.shape}")

    # 贪心解码
    model.eval()
    start_symbol = 1
    end_symbol = 2
    max_len = 10

    translated = translator.greedy_decode(src, src_mask, max_len, start_symbol, end_symbol)

    print(f"翻译结果形状: {translated.shape}")
    print(f"翻译结果: {translated}")

    return translator

In [ ]:
def analyze_transformer_components():
    """分析Transformer各组件"""
    print("\n=== Transformer组件分析 ===")

    analysis = {
        "核心创新": {
            "多头注意力": "并行计算多个注意力子空间",
            "位置编码": "无递归结构下的位置信息建模",
            "残差连接": "解决深层网络训练问题",
            "RMSNorm归一化": "现代化的高效归一化方法"
        },

        "架构优势": {
            "并行化": "相比RNN，注意力机制可完全并行",
            "长距离依赖": "直接建模任意位置间的关系",
            "可解释性": "注意力权重提供可解释性",
            "可扩展性": "易于扩展到更大模型"
        },

        "数学公式": {
            "注意力": "Attention(Q,K,V) = softmax(QK^T/√d_k)V",
            "多头": "MultiHead(Q,K,V) = Concat(head_1,...,head_h)W^O",
            "位置编码": "PE(pos,2i) = sin(pos/10000^(2i/d_model))",
            "前馈网络": "FFN(x) = max(0,xW_1+b_1)W_2+b_2",
            "RMSNorm": "RMSNorm(x) = x / RMS(x) * γ"
        },

        "现代化改进": {
            "RMSNorm": "去掉均值计算，提升效率",
            "计算优化": "减少约10-15%的归一化时间",
            "内存友好": "降低约5-10%的内存占用",
            "数值稳定": "避免均值计算的数值问题"
        },

        "训练技巧": {
            "标签平滑": "防止过拟合，提高泛化能力",
            "学习率预热": "先增后减的学习率调度",
            "残差dropout": "在残差连接前应用dropout",
            "梯度裁剪": "防止梯度爆炸"
        }
    }

    for category, details in analysis.items():
        print(f"\n{category}:")
        for key, value in details.items():
            print(f"  {key}: {value}")

    return analysis

In [ ]:
def compare_with_modern_llms():
    """与现代LLM对比"""
    print("\n=== 与现代LLM对比 ===")

    comparison = {
        "架构演进": {
            "经典Transformer": "Encoder-Decoder架构",
            "BERT": "仅Encoder，双向注意力",
            "GPT": "仅Decoder，单向注意力",
            "T5": "Encoder-Decoder + 相对位置编码",
            "现代LLM": "主要基于Decoder-only架构"
        },

        "归一化演进": {
            "经典Transformer": "Post-norm LayerNorm",
            "GPT-2": "Pre-norm LayerNorm",
            "T5": "RMSNorm",
            "LLaMA": "RMSNorm + Pre-norm",
            "PaLM": "RMSNorm + 并行层"
        },

        "位置编码演进": {
            "经典Transformer": "固定正弦余弦编码",
            "BERT/GPT": "可学习的绝对位置嵌入",
            "T5": "相对位置注意力偏置",
            "RoPE": "旋转位置编码（LLaMA使用）",
            "ALiBi": "线性偏置注意力"
        },

        "激活函数演进": {
            "经典Transformer": "ReLU",
            "BERT/GPT": "GELU",
            "现代LLM": "SwiGLU, GeGLU等门控激活"
        },

        "RMSNorm应用": {
            "T5 (2019)": "首次在大规模模型中使用RMSNorm",
            "LLaMA (2023)": "RMSNorm + SwiGLU + RoPE",
            "PaLM (2022)": "RMSNorm + 并行层设计",
            "优势": "计算效率提升10-15%，内存占用降低5-10%"
        },

        "规模演进": {
            "经典Transformer": "~65M参数",
            "BERT-Base": "110M参数",
            "GPT-3": "175B参数",
            "PaLM": "540B参数",
            "GPT-4": "估计1.8T参数"
        }
    }

    for category, details in comparison.items():
        print(f"\n{category}:")
        for model, description in details.items():
            print(f"  {model}: {description}")

    return comparison

In [ ]:
def demonstrate_attention_patterns():
    """演示注意力模式"""
    print("\n=== 注意力模式演示 ===")

    # 创建简单的注意力示例
    d_model = 64
    n_head = 4
    seq_len = 8

    attention = MultiHeadAttention(d_model, n_head, dropout=0.0)

    # 创建有意义的输入序列（模拟词嵌入）
    torch.manual_seed(42)
    x = torch.randn(1, seq_len, d_model)

    # 计算注意力
    output, attn_weights = attention(x, x, x)

    print(f"注意力权重矩阵 (第一个头):")
    first_head_attn = attn_weights[0, 0].detach().numpy()

    # 打印注意力矩阵（格式化输出）
    print("位置:", end="")
    for j in range(seq_len):
        print(f"{j:6}", end="")
    print()

    for i in range(seq_len):
        print(f"  {i}:", end="")
        for j in range(seq_len):
            print(f"{first_head_attn[i,j]:6.3f}", end="")
        print()

    # 分析注意力模式
    print(f"\n注意力分析:")
    print(f"- 对角线注意力 (自注意): {first_head_attn.diagonal().mean():.3f}")
    print(f"- 注意力分布熵: {-np.sum(first_head_attn * np.log(first_head_attn + 1e-10), axis=1).mean():.3f}")

    return attn_weights

In [ ]:
def benchmark_transformer_performance():
    """基准测试Transformer性能"""
    print("\n=== Transformer性能基准测试 ===")

    import time

    configs = [
        ("Small", TransformerConfig(d_model=256, n_head=4, n_layers=2, d_ff=512)),
        ("Base", TransformerConfig(d_model=512, n_head=8, n_layers=6, d_ff=2048)),
        ("Large", TransformerConfig(d_model=768, n_head=12, n_layers=12, d_ff=3072)),
    ]

    batch_size = 4
    seq_len = 64

    results = {}

    for name, config in configs:
        model = Transformer(config)
        model.eval()

        # 准备测试数据
        src = torch.randint(1, config.src_vocab_size, (batch_size, seq_len))
        tgt = torch.randint(1, config.tgt_vocab_size, (batch_size, seq_len))

        # 预热
        with torch.no_grad():
            _ = model(src, tgt[:, :-1])

        # 计时测试
        num_runs = 10
        start_time = time.time()

        with torch.no_grad():
            for _ in range(num_runs):
                _ = model(src, tgt[:, :-1])

        end_time = time.time()
        avg_time = (end_time - start_time) / num_runs

        param_count = sum(p.numel() for p in model.parameters())

        results[name] = {
            "参数数量": f"{param_count:,}",
            "平均推理时间": f"{avg_time:.4f}s",
            "每秒处理token": f"{(batch_size * seq_len / avg_time):.0f}"
        }

        print(f"{name} Transformer (RMSNorm):")
        for metric, value in results[name].items():
            print(f"  {metric}: {value}")
        print()

    return results

In [ ]:
if __name__ == "__main__":
    print("🚀 现代化Transformer实现 - 'Attention Is All You Need' + RMSNorm")
    print("=" * 70)

    # 设置随机种子
    torch.manual_seed(42)

    # 1. 测试完整模型
    transformer_model = test_transformer_model()

    # 2. 测试注意力机制
    attention_module, attention_weights = test_attention_mechanism()

    # 3. 测试位置编码
    pos_encoding = test_positional_encoding()

    # 4. 测试RMSNorm vs LayerNorm
    rms_norm, layer_norm = test_rmsnorm_vs_layernorm()

    # 5. 测试训练过程
    trainer, training_losses = test_training_process()

    # 6. 测试翻译推理
    translator = test_translation_inference()

    # 7. 分析组件
    component_analysis = analyze_transformer_components()

    # 8. 与现代LLM对比
    llm_comparison = compare_with_modern_llms()

    # 9. 演示注意力模式
    attention_patterns = demonstrate_attention_patterns()

    # 10. 性能基准测试
    performance_results = benchmark_transformer_performance()

    print("\n" + "=" * 70)
    print("🎯 现代化Transformer实现总结")
    print("=" * 70)
    print("✅ 完整的Encoder-Decoder架构")
    print("✅ 多头注意力机制 (核心创新)")
    print("✅ 正弦余弦位置编码")
    print("✅ 残差连接 + RMSNorm (现代化归一化)")
    print("✅ 位置相关前馈网络")
    print("✅ 掩码机制（修复版本）")
    print("✅ 标签平滑损失")
    print("✅ 学习率预热调度")
    print("✅ 贪心解码推理")
    print("✅ 注意力可视化分析")
    print("✅ 性能基准测试")

    param_count = sum(p.numel() for p in transformer_model.parameters())
    print(f"\n📊 模型统计:")
    print(f"- 参数数量: {param_count:,}")
    print(f"- 论文原始模型: ~65M 参数")
    print(f"- 核心创新: 完全基于注意力机制，摒弃递归和卷积")

    print(f"\n🔬 关键技术贡献:")
    print(f"- 提出了Scaled Dot-Product Attention")
    print(f"- 证明了注意力机制的并行化优势")
    print(f"- 建立了现代Transformer架构的基础")
    print(f"- 为BERT、GPT等模型奠定了理论基础")

    print(f"\n🚀 现代化改进:")
    print(f"- 使用RMSNorm替代LayerNorm（计算更高效）")
    print(f"- RMSNorm被LLaMA、PaLM等现代模型广泛采用")
    print(f"- 减少了均值计算，提升训练和推理速度")
    print(f"- 在保持性能的同时简化了归一化过程")

    print(f"\n💡 在AI面试中的重点:")
    print(f"- 深入理解注意力机制的数学原理")
    print(f"- 掌握Encoder-Decoder架构的工作原理")
    print(f"- 了解RMSNorm相对于LayerNorm的优势")
    print(f"- 理解残差连接和归一化的作用")
    print(f"- 能够解释现代LLM相对于经典Transformer的改进")
    print(f"- 掌握掩码机制和序列到序列建模")

🚀 现代化Transformer实现 - 'Attention Is All You Need' + RMSNorm
=== 现代化Transformer模型测试（使用RMSNorm）===
源序列形状: torch.Size([2, 20])
目标序列形状: torch.Size([2, 15])
解码器输入形状: torch.Size([2, 14])
源掩码形状: torch.Size([2, 1, 1, 20])
目标掩码形状: torch.Size([2, 1, 14, 14])
模型参数数量: 45,661,160
输出形状: torch.Size([2, 14, 1000])
输出是log概率: 检查概率和是否为1...
概率和的范围: [1.000000, 1.000000]
概率和接近1: True

=== 多头注意力机制测试 ===
输入形状: torch.Size([2, 10, 512])
输出形状: torch.Size([2, 10, 512])
注意力权重形状: torch.Size([2, 8, 10, 10])
注意力权重和为1: False

=== 位置编码测试 ===
原始输入形状: torch.Size([2, 20, 512])
位置编码后形状: torch.Size([2, 20, 512])
位置编码矩阵形状: torch.Size([20, 10])
位置编码前几个位置的值:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000],
        [ 0.8415,  0.5403,  0.8219,  0.5697,  0.8020],
        [ 0.9093, -0.4161,  0.9364, -0.3509,  0.9581],
        [ 0.1411, -0.9900,  0.2451, -0.9695,  0.3428],
        [-0.7568, -0.6536, -0.6572, -0.7537, -0.5486]])

=== RMSNorm vs LayerNorm 对比测试 ===
输入形状: torch.Size([2, 10, 512])
RMSNorm输出形状: torch.Size([2, 10, 51

# RMSNorm

In [ ]:
class RMSNorm(nn.Module):
    """
    RMS Normalization - 现代LLM中常用的归一化方法
    相比LayerNorm，RMSNorm去掉了减均值的操作，只保留缩放
    公式: RMSNorm(x) = x / RMS(x) * g
    其中 RMS(x) = sqrt(mean(x^2) + eps)

    优势:
    1. 计算更高效 - 不需要计算均值和方差
    2. 数值更稳定 - 避免了减均值操作
    3. 现代模型采用 - LLaMA、PaLM等都使用RMSNorm
    """
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 计算RMS值
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps)
        # 归一化并缩放
        return x / rms * self.weight